Retrain SVC models for plots

# Libraries

In [2]:
import os
import shutil
from pathlib import Path

import joblib
import pandas as pd

In [3]:
RANDOM_STATE = 42

In [4]:
# train/test split
import sys
sys.path.insert(1, '../utils_functionality/split_utils/')
from split_tools import get_train_test

In [5]:
# dict with num features
sys.path.insert(1, '../utils_functionality/models/')
from modelling2_hyperparams import dict_num_features

# `probability=True` in SVC

In [35]:
df_metrics = pd.read_excel(r'..\results\metrics_modelling2_filtered_optuna.xlsx', index_col=[0])

- best_models_modelling_2/svc_smote_net_impact_ordenc_df_modelling_dimensionless_pf_net_impact.pkl

In [28]:
path_pipeline = r'../results/best_models_modelling_2/svc_smote_net_impact_ordenc_df_modelling_dimensionless_pf_net_impact.pkl'
pipeline = joblib.load(path_pipeline)
pipeline

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['Re^(1/4)', 'We^(1/4)^2',
                                                   'We^(1/4)^2 Re^(1/4)',
                                                   'inclination',
                                                   'particle_liquid_density_ratio',
                                                   'particle_droplet_diameter_ratio']),
                                                 ('cat',
                                                  Pipeline(steps=[('ordenc',
                                                                   OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                                  unknown_value=nan))]),
                                                  ['wettability'])])),
                ('model',
                 SVC(C=4026.5502671808963, coef0=-0.17141513324034952,
                     gamma=1.2146813942954187))])

In [32]:
pipeline.steps[1][1].probability = True
pipeline.steps[1][1].get_params()['probability']

True

# Chossing corresponding data

In [48]:
# svc_smote_net_impact_ordenc_df_modelling_dimensionless_pf_net_impact
row_model_data = df_metrics.loc[
    (df_metrics['target']=='net_impact') &
    (df_metrics['dataset']=='df_modelling_dimensionless_pf') &
    (df_metrics['model']=='svc_smote_net_impact_ordenc') &
    (df_metrics['optuna_flg']==1)
].iloc[0]
row_model_data

dataset       df_modelling_dimensionless_pf
target                           net_impact
model           svc_smote_net_impact_ordenc
accuracy                           0.946667
f1                                      0.9
precision                               0.9
recall                                  0.9
roc_auc                            0.931818
optuna_flg                              1.0
Name: 46, dtype: object

In [50]:
target = 'splashing'
train, test = get_train_test(
    dataset_filename=row_model_data['dataset'],
    target=target)
train = train[dict_num_features[row_model_data['dataset']]+['wettability']+[target]]
test = test[dict_num_features[row_model_data['dataset']]+['wettability']+[target]]

# Fitting pipeline and saving

In [56]:
pipeline.fit(X=train.drop(columns=[target]), y=train[target])
pipeline.predict_proba(test.drop(columns=[target]))[:10]

array([[0.22885888, 0.77114112],
       [0.24419206, 0.75580794],
       [0.22084309, 0.77915691],
       [0.62855167, 0.37144833],
       [0.22938134, 0.77061866],
       [0.72738253, 0.27261747],
       [0.25889169, 0.74110831],
       [0.2258426 , 0.7741574 ],
       [0.94722827, 0.05277173],
       [0.65025729, 0.34974271]])

In [57]:
path_prob_models = Path('../results/probability_models')
if not os.path.exists(path_prob_models): os.makedirs(path_prob_models)

pipeline_name = 'temp_fixed_pipeline'
joblib.dump(pipeline, str(path_prob_models / f'{pipeline_name}.pkl'))